## Scenario 2: A cross-functional team with one data scientist working on an ML model
MLflow setup:

- tracking server: yes, local server
- backend store: sqlite database
- artifacts store: local filesystem
- The experiments can be explored locally by accessing the local tracking server.

To run this example you need to launch the mlflow server locally by running the following command in your terminal:

 - mlflow server --backend-store-uri sqlite:///backend.db --default-artifact-root ./artifacts_local_scenario2

In [20]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5000'


In [21]:

mlflow.search_experiments()

[<Experiment: artifact_location='/home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/02-experiment-tracking/running_mlflow_escenarios/artifacts_local_scenario2/0', creation_time=1778472769035, experiment_id='0', last_update_time=1778472769035, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

### Registering experiments

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/05/10 23:12:59 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2026/05/10 23:12:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 23:12:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


default artifacts URI: '/home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/02-experiment-tracking/running_mlflow_escenarios/artifacts_local_scenario2/1/b91f577075324c6a87c1f4d033a2ca88/artifacts'
🏃 View run treasured-gull-565 at: http://127.0.0.1:5000/#/experiments/1/runs/b91f577075324c6a87c1f4d033a2ca88
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [23]:
mlflow.search_experiments()

[<Experiment: artifact_location='/home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/02-experiment-tracking/running_mlflow_escenarios/artifacts_local_scenario2/1', creation_time=1778472779176, experiment_id='1', last_update_time=1778472779176, lifecycle_stage='active', name='my-experiment-1', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/02-experiment-tracking/running_mlflow_escenarios/artifacts_local_scenario2/0', creation_time=1778472769035, experiment_id='0', last_update_time=1778472769035, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

### Interacting with the model registry

In [24]:
from mlflow.tracking import MlflowClient
client = MlflowClient("http://127.0.0.1:5000")

In [27]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1778472796947, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description='', last_updated_timestamp=1778472797186, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1778472797186, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1778472797186, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='b91f577075324c6a87c1f4d033a2ca88', run_link='', source='models:/m-a4bb64f4ef1e4c9d809cb0437f831b8c', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>], name='iris-classifier', tags={}, workspace='default'>]

In [26]:
run_id = client.search_runs(experiment_ids='1')[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2026/05/10 23:13:17 WARNING mlflow.tracking._model_registry.fluent: Run with id b91f577075324c6a87c1f4d033a2ca88 has no artifacts at artifact path 'models', registering model based on models:/m-a4bb64f4ef1e4c9d809cb0437f831b8c instead
2026/05/10 23:13:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1778472797186, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1778472797186, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='b91f577075324c6a87c1f4d033a2ca88', run_link='', source='models:/m-a4bb64f4ef1e4c9d809cb0437f831b8c', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>